In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "1"  # OpenMP
os.environ["MKL_NUM_THREADS"] = "1"  # MKL
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # OpenBLAS
os.environ["NUMEXPR_NUM_THREADS"] = "1"  # numexpr

In [3]:
import json
import sys
from pathlib import Path
import dask
import numpy as np
import seaborn as sns
import xarray as xr
from dask.diagnostics import ProgressBar

sys.path.append("../..")

from src.data_processing.load_data import ClimateModelData
from src.utils.constants import (
    EXTENT_BC,
    EXTENT_CH,
    BC_SUBDAILY_CHUNKS,
)
from src.utils.utils_xarray import sel_extent
from src.conversion_energy.wind import (
	extrapolate_wind_speed
)

sns.set_style("whitegrid", {"grid.color": ".93"})

In [4]:
paths = json.load(open("../../paths_so4.json"))

path_data_analysis = Path(paths["path_data_analysis"])
path_data_processed = Path(paths["path_data_processed"])
path_data_cordex = Path(paths["path_data_cordex"])
path_data_adjusted_cordex = Path(paths["path_data_adjusted_cordex"])
path_data_cerra = Path(paths["path_data_cerra"])
path_data_cerra_land = Path(paths["path_data_cerra_land"])
path_data_personal = Path(paths["path_data_personal"])
path_data_personal_cordex = path_data_personal / "cordex"
path_data_temporary = Path(paths["path_data_temporary"])

path_energy_ch = path_data_processed / "energy" / "ch"
path_energy_europe = path_data_processed / "energy" / "europe"

path_figs = Path(paths["path_figs"])

output_dir = Path("/net/exo/landclim_nobackup/FTP/yhaddad/")

In [5]:
dask.config.set(num_workers=15, threads_per_worker=16)
dask.config.set(
    {
        "distributed.worker.memory.target": 0.7,  # Spill to disk at 70%
        "distributed.worker.memory.spill": 0.8,  # Start spilling at 80%
        "distributed.worker.memory.pause": 0.9,  # Pause worker at 90%
        "array.chunk-size": "512MiB",
        "logging.distributed": "error",  # Less verbose logs
    }
)

In [6]:
climate_models = [
    "CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp26",
    "CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp45",
    "CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp85",
]

## PV

In [15]:
for cm in climate_models:
    print(f"Processing {cm}...")
    output_path = (output_dir / "pv_ch" / f"pv_{cm}_all_tilts_azimuths.nc")
    if output_path.exists():
        continue
    else:
        with ProgressBar(dt=1):
            xr.open_zarr(list((path_energy_ch / "pv" / cm).glob("*.zarr"))[0]).sel(time=slice("1991", "2052")).to_netcdf(
                output_path
            )

Processing CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp26...
Processing CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp45...
[########################################] | 100% Completed | 108.23 s
Processing CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp85...


# Wind

In [11]:
for cm in climate_models:
    print(f"Processing {cm}...")
    output_path = (output_dir / "wind_ch" / "wind_speed" / f"wind_{cm}.nc")
    if output_path.exists():
        continue
    else:
        path_wind_100m = (
                path_data_processed
                / "cordex_processed"
                / f"{cm}"
                / "bias_corrected"
                / "3hr"
                / "wind100m.zarr"
            )
        if path_wind_100m.exists():
            output_path_wind100m = (output_dir / "wind_ch" / "wind_speed" / f"wind100m_{cm}.nc")
            output_path_wind150m = (output_dir / "wind_ch" / "wind_speed" / f"wind150m_{cm}.nc")
            if not (output_path_wind100m.exists() and output_path_wind150m.exists()):
                with ProgressBar(dt=1):
                    sel_extent(xr.open_zarr(path_wind_100m, chunks="auto"), extent=EXTENT_CH).sel(time=slice("2018", "2052")).to_netcdf(
                        output_path_wind100m
                    )
                    sel_extent(xr.open_zarr(path_wind_100m.parent / "wind150m.zarr", chunks="auto"), extent=EXTENT_CH).sel(time=slice("2018", "2052")).to_netcdf(
                        output_path_wind150m
                    )

        else:
            heights = (10, 100)
            output_path_sfcWind = (output_dir / "wind_ch" / "wind_speed" / f"wind10m_{cm}.nc")
            output_path_wind100m = (output_dir / "wind_ch" / "wind_speed" / f"wind100m_{cm}.nc")
            if not (output_path_sfcWind.exists() and output_path_wind100m.exists()):
                with ProgressBar(dt=1):
                    da_alpha = sel_extent(
                        xr.open_dataset(
                            path_data_processed
                            / "cerra_processed"
                            / "alpha"
                            / f"da_alpha_temp_smoothed_cerra_heights_{heights[0]}_{heights[1]}.nc"
                        ).alpha,
                        extent=EXTENT_CH,
                    )
                    ds_wind = ClimateModelData(
                        path_wind_100m.parent / "sfcWind.zarr",
                        extent=EXTENT_CH,
                        chunks="auto",
                    )
                    ds_wind.ds_data = ds_wind.ds_data.sel(time=slice("2018", "2052"))
                    da_alpha = da_alpha.isel(
                        time=(np.arange(len(ds_wind.ds_data.time)) % len(da_alpha.time))
                    ).assign_coords(time=ds_wind.ds_data.time).chunk(BC_SUBDAILY_CHUNKS)
                    wind_var = list(ds_wind.ds_data.data_vars.keys())[0]

                    ds_wind.ds_data.to_netcdf(output_path_sfcWind)
                    extrapolate_wind_speed(
                        ds_wind.ds_data[wind_var], heights[0], 100, da_alpha
                    ).rename("wind100m").to_dataset().to_netcdf(output_path_wind100m)


Processing CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp26...
Processing CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp45...
[########################################] | 100% Completed | 172.18 s
[########################################] | 100% Completed | 233.21 s
Processing CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1_rcp85...
[########################################] | 100% Completed | 5.03 ss
[########################################] | 100% Completed | 7.18 ss
